# Download a dataset

Downloads a dataset from R2 into `datasets/` only if it's missing, then loads it with pandas.

**Setup:** copy `.env.example` to `.env`, fill in your R2 values, and select the project's `.venv` as the notebook kernel.

In [ ]:
from project_argus.datasets import DATASETS, fetch_dataset

print("Available:", DATASETS)

DATASET = "DS_Augmented_v2.csv"
path = fetch_dataset(DATASET)  # Shows a progress bar when it downloads.
print("Ready:", path)

## Load a small sample

These are the first rows of the file, not a random sample, so they may not include every class. The options keep text such as `NA` or `null` intact.

In [ ]:
import pandas as pd

READ_OPTIONS = {"dtype": str, "keep_default_na": False}
COLUMN_NAMES = {"col1": "request", "col2": "label", "Full_request": "request", "class": "label"}

sample = pd.read_csv(path, nrows=1000, **READ_OPTIONS).rename(columns=COLUMN_NAMES)
print(sample.shape)
sample["request"].str.len().describe()

## Count the classes

Reads the file in chunks, so this works even when the whole file wouldn't fit in memory.

In [ ]:
from collections import Counter

labels = Counter()
for chunk in pd.read_csv(path, usecols=[1], chunksize=100_000, **READ_OPTIONS):
    labels.update(chunk.iloc[:, 0].value_counts().to_dict())

print("Rows:", sum(labels.values()))
pd.Series(labels).sort_values(ascending=False)

## Load the whole file

Uncomment this when you're ready. It can take several GB of RAM, so check your free memory first. Avoid saving notebook outputs that show requests, because they can contain attack payloads and cookies.

In [ ]:
# df = pd.read_csv(path, **READ_OPTIONS).rename(columns=COLUMN_NAMES)
# df.shape